In [ ]:
!pip install optuna scikit-learn pillow

In [ ]:
!pip install torch torchvision

In [ ]:
import os
import time
import random
import warnings
import logging
from pathlib import Path
from typing import List, Tuple, Dict, Optional, Any
from dataclasses import dataclass, field
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image, UnidentifiedImageError
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    classification_report,
    roc_curve,
)
from sklearn.model_selection import train_test_split

import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

In [ ]:
SEED = 42

def set_global_seed(seed: int) -> None:
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  os.environ["PYTHONHASHSEED"] = str(seed)

set_global_seed(SEED)

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False   # allow faster non-deterministic ops
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")
print(f"torchvision  : {torchvision.__version__}")
if DEVICE.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
REPO_URL  = "https://github.com/jordan-bird/CIFAKE-Real-and-AI-Generated-Synthetic-Images.git"
REPO_NAME = "CIFAKE-Real-and-AI-Generated-Synthetic-Images"

if not Path(REPO_NAME).exists():
    print("Cloning CIFAKE repository …")
    os.system(f"git clone --depth=1 {REPO_URL}")   # --depth=1 = shallow clone (faster)
    print("Clone complete.")
else:
    print("📁 Repository already present — skipping clone.")

DATASET_ROOT = Path(f"{REPO_NAME}/DATASET")


expected_dirs = [
    DATASET_ROOT / "train" / "REAL",
    DATASET_ROOT / "train" / "FAKE",
    DATASET_ROOT / "test"  / "REAL",
    DATASET_ROOT / "test"  / "FAKE",
]

for d in expected_dirs:
    assert d.exists(), f"Expected directory missing: {d}"

print("Dataset structure verified.")
print(f"   📂 Root : {DATASET_ROOT.resolve()}")

# Quick file count per split / class
for split in ("train", "test"):
    for cls in ("REAL", "FAKE"):
        n = len(list((DATASET_ROOT / split / cls).glob("*")))
        print(f"   {split:5s} / {cls:4s} : {n:>6,} images")

In [ ]:
@dataclass
class PathConfig:
    """All filesystem paths used in this project."""
    dataset_root  : Path = Path("CIFAKE-Real-and-AI-Generated-Synthetic-Images/DATASET")
    train_dir     : Path = field(init=False)
    test_dir      : Path = field(init=False)
    checkpoint_dir: Path = Path("checkpoints")
    results_dir   : Path = Path("results")

    def __post_init__(self):
        self.train_dir = self.dataset_root / "train"
        self.test_dir  = self.dataset_root / "test"
        # Create output directories so saves don't fail later
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.results_dir.mkdir(parents=True, exist_ok=True)


@dataclass
class ImageConfig:
    """Settings that control how images are loaded and pre-processed."""
    image_size     : int  = 32        # CIFAKE images are 32×32
    num_channels   : int  = 3
    min_image_size : int  = 16
    force_rgb      : bool = True
    # ImageNet mean/std — used when fine-tuning pretrained ResNets
    imagenet_mean  : Tuple = (0.485, 0.456, 0.406)
    imagenet_std   : Tuple = (0.229, 0.224, 0.225)
    # Simple symmetric normalization for custom CNN
    cifake_mean: Tuple = (0.5, 0.5, 0.5)
    cifake_std : Tuple = (0.5, 0.5, 0.5)

In [ ]:
import os
import time
import random
import warnings
import logging
from pathlib import Path
from typing import List, Tuple, Dict, Optional, Any
from dataclasses import dataclass, field
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image, UnidentifiedImageError
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    classification_report,
    roc_curve,
)
from sklearn.model_selection import train_test_split

import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

@dataclass
class TrainConfig:
    batch_size        : int   = 128
    num_epochs        : int   = 20
    learning_rate     : float = 1e-3
    weight_decay      : float = 1e-4
    num_workers       : int   = 0
    val_fraction      : float = 0.20
    early_stop_patience: int  = 5
    label_smoothing   : float = 0.05
    # Scheduler — we use CosineAnnealing over the full run
    scheduler_T_max   : int   = field(init=False)

    def __post_init__(self):
        self.scheduler_T_max = self.num_epochs

@dataclass
class OptunaConfig:
    """Hyperparameter search settings."""
    n_trials          : int   = 10
    tuning_epochs     : int   = 8
    sampler           : str   = "TPE"
    pruner            : str   = "Median"
    direction         : str   = "maximize"
    study_name        : str   = "cifake_cnn_search"


@dataclass
class EfficiencyConfig:
    """Settings for model size and inference speed benchmarking."""
    benchmark_iterations : int = 100
    benchmark_batch_size : int = 1
    warmup_iterations    : int = 5


PATH_CFG      = PathConfig()
IMG_CFG       = ImageConfig()
TRAIN_CFG     = TrainConfig()
OPTUNA_CFG    = OptunaConfig()
EFFICIENCY_CFG = EfficiencyConfig()

CLASS_TO_IDX  = {"REAL": 0, "FAKE": 1}
IDX_TO_CLASS  = {v: k for k, v in CLASS_TO_IDX.items()}
CLASS_NAMES   = ["REAL", "FAKE"]
NUM_CLASSES   = 2

print("Configuration objects created.")
pprint({
    "Batch size"    : TRAIN_CFG.batch_size,
    "Epochs"        : TRAIN_CFG.num_epochs,
    "LR"            : TRAIN_CFG.learning_rate,
    "Optuna trials" : OPTUNA_CFG.n_trials,
    "Image size"    : IMG_CFG.image_size,
    "Device"        : str(DEVICE),
})

In [ ]:
def scan_directory(root: Path, class_to_idx: Dict[str, int]) -> List[Tuple[Path, int]]:
    samples = []

    for class_name, label in class_to_idx.items():
        class_dir = root / class_name
        if not class_dir.is_dir():
            print(f"Class directory not found: {class_dir}")
            continue

        for file_path in sorted(class_dir.iterdir()):
            if file_path.is_file():
                samples.append((file_path, label))

    return samples

print("Scanning training directory …")
train_samples_all = scan_directory(PATH_CFG.train_dir, CLASS_TO_IDX)

print("Scanning test directory …")
test_samples = scan_directory(PATH_CFG.test_dir, CLASS_TO_IDX)

print(f"\nDataset summary")
print(f"   Full train pool : {len(train_samples_all):>7,} images")
print(f"   Test set        : {len(test_samples):>7,} images")
print(f"   Total           : {len(train_samples_all) + len(test_samples):>7,} images")


def get_class_distribution(samples: List[Tuple[Path, int]]) -> Dict[str, int]:
    """Count samples per class from a (path, label) list."""
    distribution = {name: 0 for name in CLASS_TO_IDX}
    idx_to_name  = {v: k for k, v in CLASS_TO_IDX.items()}
    for _, label in samples:
        distribution[idx_to_name[label]] += 1
    return distribution


train_dist = get_class_distribution(train_samples_all)
test_dist  = get_class_distribution(test_samples)

print("\nClass distribution (train pool):")
for cls, count in train_dist.items():
    pct = 100 * count / len(train_samples_all)
    print(f"   {cls:4s} : {count:>6,}  ({pct:.1f} %)")

print("\Class distribution (test):")
for cls, count in test_dist.items():
    pct = 100 * count / len(test_samples)
    print(f"   {cls:4s} : {count:>6,}  ({pct:.1f} %)")

In [ ]:
class SafeImageDataset(Dataset):
    def __init__(
        self,
        samples: List[Tuple[Path, int]],
        transform: Optional[Any] = None,
        img_cfg: ImageConfig = IMG_CFG,
        split_name: str = "dataset",
        max_retries: int = 10,
    ):
        self.transform = transform
        self.img_cfg = img_cfg
        self.split_name = split_name
        self.max_retries = max_retries

        self._errors: Dict[str, List[str]] = {
            "verify_failed": [],
            "load_failed": [],
            "too_small": [],
            "mode_issue": [],
        }

        self.samples: List[Tuple[Path, int]] = []
        self._verify_samples(samples)

        print(
            f"[{split_name}] {len(self.samples):,} valid / "
            f"{len(samples):,} total ({len(samples) - len(self.samples)} skipped)"
        )

    def _verify_samples(self, raw_samples: List[Tuple[Path, int]]) -> None:
        for path, label in raw_samples:
            try:
                with Image.open(path) as img:
                    img.verify()
                self.samples.append((path, label))
            except Exception as exc:
                self._errors["verify_failed"].append(f"{path} -> {exc}")

    def _load_image(self, path: Path) -> Image.Image:
        try:
            with Image.open(path) as img:
                if self.img_cfg.force_rgb and img.mode != "RGB":
                    try:
                        img = img.convert("RGB")
                    except Exception as exc:
                        self._errors["mode_issue"].append(f"{path} -> {exc}")
                        raise RuntimeError(f"Mode conversion failed: {exc}")

                w, h = img.size
                if w < self.img_cfg.min_image_size or h < self.img_cfg.min_image_size:
                    self._errors["too_small"].append(
                        f"{path} -> {w}x{h} below minimum {self.img_cfg.min_image_size}"
                    )
                    raise RuntimeError(f"Image too small: {w}x{h}")

                return img.copy()

        except RuntimeError:
            raise
        except Exception as exc:
            self._errors["load_failed"].append(f"{path} -> {exc}")
            raise RuntimeError(f"Load failed: {exc}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        for attempt in range(self.max_retries):
            current_idx = (idx + attempt) % len(self.samples)
            path, label = self.samples[current_idx]

            try:
                img = self._load_image(path)
                if self.transform is not None:
                    img = self.transform(img)
                return img, label

            except RuntimeError:
                continue

        raise RuntimeError(
            f"[{self.split_name}] Failed to load valid image after "
            f"{self.max_retries} retries starting from index {idx}"
        )

    def get_error_summary(self) -> None:
        total = sum(len(v) for v in self._errors.values())
        print(f"\nError summary for '{self.split_name}' ({total} total issues):")

        for category, entries in self._errors.items():
            if entries:
                print(f"  {category:20s}: {len(entries)}")
                for e in entries[:3]:
                    print(f"    -> {e}")
                if len(entries) > 3:
                    print(f"    ... and {len(entries) - 3} more")

        if total == 0:
            print("  No errors - dataset looks clean.")

In [ ]:
train_samples_all[0]

In [ ]:
labels_all = [label for _, label in train_samples_all]
labels_all.count(0),labels_all.count(1)

In [ ]:
train_samples, val_samples = train_test_split(
    train_samples_all,
    test_size   = TRAIN_CFG.val_fraction,
    stratify    = labels_all,
    random_state= SEED,
)
print("Split sizes:")
print(f"   Train      : {len(train_samples):>6,}")
print(f"   Validation : {len(val_samples):>6,}")
print(f"   Test       : {len(test_samples):>6,}")

In [ ]:
#verify the balance
def verify_split_balance(samples: List[Tuple[Path, int]], name: str) -> None:
    """Print class balance for a given sample list."""
    labels  = [l for _, l in samples]
    for label, cls in IDX_TO_CLASS.items():
        count = labels.count(label)
        pct   = 100 * count / len(labels)
        print(f"{name:12s} | {cls:4s} : {count:>6,}  ({pct:.1f} %)")

print("\nClass balance after split:")
verify_split_balance(train_samples, "train")
verify_split_balance(val_samples,   "val")
verify_split_balance(test_samples,  "test")

In [ ]:
imbalance_ratio = max(train_dist.values()) / min(train_dist.values())
print(f"Train imbalance ratio: {imbalance_ratio:.2f}")

In [ ]:
CNN_IMG_SIZE = IMG_CFG.image_size

cnn_train_transform = transforms.Compose([
    transforms.Resize((CNN_IMG_SIZE, CNN_IMG_SIZE)),
    transforms.RandomCrop(CNN_IMG_SIZE, padding=2),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_CFG.cifake_mean, std=IMG_CFG.cifake_std),
])

cnn_eval_transform = transforms.Compose([
    transforms.Resize((CNN_IMG_SIZE, CNN_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_CFG.cifake_mean, std=IMG_CFG.cifake_std),
])



RESNET_IMG_SIZE = 64

resnet_train_transform = transforms.Compose([
    transforms.Resize((RESNET_IMG_SIZE, RESNET_IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_CFG.imagenet_mean, std=IMG_CFG.imagenet_std),
])

resnet_eval_transform = transforms.Compose([
    transforms.Resize((RESNET_IMG_SIZE, RESNET_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_CFG.imagenet_mean, std=IMG_CFG.imagenet_std),
])

print("Transform pipelines defined.")
print(f"Custom CNN input size : {CNN_IMG_SIZE}x{CNN_IMG_SIZE}")
print(f"ResNet input size     : {RESNET_IMG_SIZE}x{RESNET_IMG_SIZE}")

In [ ]:
def build_loaders(
    train_samples : List[Tuple[Path, int]],
    val_samples   : List[Tuple[Path, int]],
    test_samples  : List[Tuple[Path, int]],
    train_tf      : transforms.Compose,
    eval_tf       : transforms.Compose,
    batch_size    : int = TRAIN_CFG.batch_size,
    num_workers   : int = TRAIN_CFG.num_workers,
    split_prefix  : str = "",
) -> Dict[str, DataLoader]:

    train_ds = SafeImageDataset(train_samples, train_tf, split_name=f"{split_prefix}train")
    val_ds   = SafeImageDataset(val_samples,   eval_tf,  split_name=f"{split_prefix}val")
    test_ds  = SafeImageDataset(test_samples,  eval_tf,  split_name=f"{split_prefix}test")

    use_pin_memory = (DEVICE.type == "cuda")

    train_loader = DataLoader(
        train_ds,
        batch_size        = batch_size,
        shuffle           = True,
        num_workers       = num_workers,
        pin_memory        = use_pin_memory,
        persistent_workers= (num_workers > 0),
        drop_last         = True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size        = batch_size,
        shuffle           = False,
        num_workers       = num_workers,
        pin_memory        = use_pin_memory,
        persistent_workers= (num_workers > 0),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size        = batch_size,
        shuffle            = False,
        num_workers       = num_workers,
        pin_memory        = use_pin_memory,
        persistent_workers= (num_workers > 0),
    )

    return {"train": train_loader, "val": val_loader, "test": test_loader}


print("\nBuilding CNN DataLoaders …")
cnn_loaders = build_loaders(
    train_samples, val_samples, test_samples,
    cnn_train_transform, cnn_eval_transform,
    split_prefix="cnn_",
)

print("\nBuilding ResNet DataLoaders …")
resnet_loaders = build_loaders(
    train_samples, val_samples, test_samples,
    resnet_train_transform, resnet_eval_transform,
    split_prefix="resnet_",
)

print("\nLoader sizes:")
for name, loader in cnn_loaders.items():
    print(f"   CNN    {name:5s} : {len(loader.dataset):>6,} samples, "
          f"{len(loader):>4} batches")
for name, loader in resnet_loaders.items():
    print(f"   ResNet {name:5s} : {len(loader.dataset):>6,} samples, "
          f"{len(loader):>4} batches")

In [ ]:
#visuals
def show_sample_images(samples, n_per_cls=5, title="Sample Images"):

    import random
    random.shuffle(samples)

    cls_samples = {k: [] for k in IDX_TO_CLASS.keys()}

    for path, label in samples:
        if len(cls_samples[label]) < n_per_cls:
            cls_samples[label].append(path)

    n_classes = len(cls_samples)

    fig, axes = plt.subplots(n_classes, n_per_cls, figsize=(2.5 * n_per_cls, 3 * n_classes))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for row, label in enumerate(cls_samples.keys()):
        for col, img_path in enumerate(cls_samples[label]):

            ax = axes[row][col] if n_classes > 1 else axes[col]

            try:
                img = Image.open(img_path).convert("RGB")
                ax.imshow(img)
            except:
                ax.set_facecolor("#eee")
                ax.text(0.5, 0.5, "ERR", ha="center", va="center", transform=ax.transAxes)

            ax.axis("off")

            if col == 0:
                ax.set_ylabel(
                    IDX_TO_CLASS[label],
                    fontsize=12,
                    fontweight="bold",
                    rotation=0,
                    labelpad=40,
                    va="center"
                )

    plt.tight_layout()
    plt.show()

In [ ]:
class FlexibleCNN(nn.Module):
    def __init__(
        self,
        in_channels: int = 3,
        conv_configs: Optional[List[Tuple[int, int]]] = None,
        dropout_rate: float = 0.3,
        fc_hidden_dim: int = 256,
        num_classes: int = 2,
        image_size: int = 32,
    ):
        super().__init__()

        if conv_configs is None:
            conv_configs = [(32, 3), (64, 3), (128, 3)]

        if len(conv_configs) not in [3, 4]:
            raise ValueError("This PetCNN-style FlexibleCNN supports only 3 or 4 conv blocks.")

        self.conv_configs = conv_configs
        self.dropout_rate = dropout_rate
        self.fc_hidden_dim = fc_hidden_dim
        self.num_classes = num_classes
        self.image_size = image_size
        self.in_channels = in_channels
        self.num_blocks = len(conv_configs)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout2d = nn.Dropout2d(dropout_rate)
        self.dropout = nn.Dropout(dropout_rate)

        out1, k1 = conv_configs[0]
        self.conv1 = nn.Conv2d(
            in_channels,
            out1,
            kernel_size=k1,
            padding=k1 // 2,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out1)


        out2, k2 = conv_configs[1]
        self.conv2 = nn.Conv2d(
            out1,
            out2,
            kernel_size=k2,
            padding=k2 // 2,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out2)

        out3, k3 = conv_configs[2]
        self.conv3 = nn.Conv2d(
            out2,
            out3,
            kernel_size=k3,
            padding=k3 // 2,
            bias=False,
        )
        self.bn3 = nn.BatchNorm2d(out3)

        if self.num_blocks == 4:
            out4, k4 = conv_configs[3]
            self.conv4 = nn.Conv2d(
                out3,
                out4,
                kernel_size=k4,
                padding=k4 // 2,
                bias=False,
            )
            self.bn4 = nn.BatchNorm2d(out4)
            last_channels = out4
        else:
            self.conv4 = None
            self.bn4 = None
            last_channels = out3

        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, image_size, image_size)
            x = self.pool(F.relu(self.bn1(self.conv1(dummy))))
            x = self.dropout2d(x)

            x = self.pool(F.relu(self.bn2(self.conv2(x))))
            x = self.dropout2d(x)

            x = self.pool(F.relu(self.bn3(self.conv3(x))))
            x = self.dropout2d(x)

            if self.num_blocks == 4:
                x = self.pool(F.relu(self.bn4(self.conv4(x))))
                x = self.dropout2d(x)

            self.flat_dim = x.view(1, -1).shape[1]


        self.fc1 = nn.Linear(self.flat_dim, fc_hidden_dim)
        self.fc2 = nn.Linear(fc_hidden_dim, num_classes)

        # save config
        self.config = {
            "in_channels": in_channels,
            "conv_configs": conv_configs,
            "dropout_rate": dropout_rate,
            "fc_hidden_dim": fc_hidden_dim,
            "num_classes": num_classes,
            "image_size": image_size,
            "flat_dim": self.flat_dim,
        }


        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # block1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout2d(x)

        # block2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout2d(x)

        # block3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.dropout2d(x)

        # optional block4
        if self.num_blocks == 4:
            x = self.pool(F.relu(self.bn4(self.conv4(x))))
            x = self.dropout2d(x)

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

    def describe(self):
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)

        print("\nFlexibleCNN Summary")
        print("-" * 50)
        print(f"Conv blocks   : {self.num_blocks}")
        for i, (ch, k) in enumerate(self.conv_configs, start=1):
            print(f"Block {i:>2}     : {ch:>4} filters, {k}x{k} kernel")
        print(f"Flat dim      : {self.flat_dim}")
        print(f"FC hidden dim : {self.fc_hidden_dim}")
        print(f"Dropout       : {self.dropout_rate}")
        print(f"Num classes   : {self.num_classes}")
        print(f"Total params  : {total_params:,}")
        print(f"Trainable     : {trainable_params:,}")

In [ ]:
# cnn_4block = FlexibleCNN(
#     in_channels=IMG_CFG.num_channels,
#     conv_configs=[(32, 3), (64, 3), (128, 3), (256, 3)],
#     dropout_rate=0.30,
#     fc_hidden_dim=256,
#     num_classes=NUM_CLASSES,
#     image_size=IMG_CFG.image_size,
# ).to(DEVICE)

# cnn_4block.describe()

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    scaler: Optional[torch.cuda.amp.GradScaler] = None,
    grad_clip: Optional[float] = 5.0,
) -> Tuple[float, float]:

    model.train()
    running_loss = 0.0
    correct_samples = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()

            if grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

            scaler.step(optimizer)
            scaler.update()

        else:
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

            optimizer.step()

        preds = logits.argmax(dim=1)
        correct_samples += (preds == labels).sum().item()
        total_samples += labels.size(0)
        running_loss += loss.item() * labels.size(0)

    avg_loss = running_loss / total_samples
    accuracy = correct_samples / total_samples

    return avg_loss, accuracy

In [ ]:
@torch.no_grad()
def evaluate_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, float, np.ndarray, np.ndarray, np.ndarray]:

    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(dim=1)

        running_loss += loss.item() * labels.size(0)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)
    accuracy = accuracy_score(all_labels, all_preds)

    return avg_loss, accuracy, all_preds, all_labels, all_probs

In [ ]:
def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_probs: np.ndarray,
) -> Dict[str, float]:

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    try:
        metrics["roc_auc"] = roc_auc_score(y_true, y_probs)
    except Exception:
        metrics["roc_auc"] = np.nan

    return metrics

In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = TRAIN_CFG.num_epochs,
    learning_rate: float = TRAIN_CFG.learning_rate,
    weight_decay: float = TRAIN_CFG.weight_decay,
    patience: int = TRAIN_CFG.early_stop_patience,
    label_smoothing: float = TRAIN_CFG.label_smoothing,
    device: torch.device = DEVICE,
    checkpoint_path: Optional[Path] = None,
    verbose: bool = True,
    grad_clip: Optional[float] = 5.0,
) -> Tuple[nn.Module, Dict[str, Any]]:

    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=1e-6,
    )

    scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = 1
    patience_ctr = 0
    t_start = time.time()

    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            grad_clip=grad_clip,
        )

        vl_loss, vl_acc, _, _, _ = evaluate_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            lr_now = scheduler.get_last_lr()[0]
            print(
                f"Epoch {epoch:>3}/{num_epochs} | "
                f"train_loss={tr_loss:.4f} | train_acc={tr_acc:.4f} | "
                f"val_loss={vl_loss:.4f} | val_acc={vl_acc:.4f} | "
                f"lr={lr_now:.2e}"
            )

        if vl_loss < best_val_loss - 1e-4:
            best_val_loss = vl_loss
            best_val_acc = vl_acc
            best_epoch = epoch
            patience_ctr = 0

            if checkpoint_path is not None:
                torch.save(model.state_dict(), checkpoint_path)
        else:
            patience_ctr += 1

            if patience_ctr >= patience:
                if verbose:
                    print(
                        f"Early stop at epoch {epoch} "
                        f"(best val loss={best_val_loss:.4f}, "
                        f"best val acc={best_val_acc:.4f}, "
                        f"best epoch={best_epoch})"
                    )
                break

    if checkpoint_path is not None and checkpoint_path.exists():
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    history["best_val_loss"] = best_val_loss
    history["best_val_acc"] = best_val_acc
    history["best_epoch"] = best_epoch
    history["training_time_s"] = time.time() - t_start

    return model, history

In [ ]:
print("=" * 65)
print("  Training Baseline FlexibleCNN")
print("=" * 65)

baseline_cnn = FlexibleCNN(
    in_channels   = IMG_CFG.num_channels,
    conv_configs  = [(32, 3), (64, 3), (128, 3)],
    dropout_rate  = 0.3,
    fc_hidden_dim = 256,
    num_classes   = NUM_CLASSES,
    image_size    = IMG_CFG.image_size,
)

baseline_cnn.describe()

In [ ]:
cnn_baseline_ckpt = PATH_CFG.checkpoint_dir / "cnn_baseline.pt"
print("Checkpoint path:", cnn_baseline_ckpt)

In [ ]:
print("=" * 65)
print("  Training Baseline FlexibleCNN")
print("=" * 65)

baseline_cnn = FlexibleCNN(
    in_channels   = IMG_CFG.num_channels,
    conv_configs  = [(32, 3), (64, 3), (128, 3)],
    dropout_rate  = 0.3,
    fc_hidden_dim = 256,
    num_classes   = NUM_CLASSES,
    image_size    = IMG_CFG.image_size,
)

baseline_cnn.describe()

cnn_baseline_ckpt = PATH_CFG.checkpoint_dir / "cnn_baseline.pt"

baseline_cnn, baseline_history = train_model(
    model           = baseline_cnn,
    train_loader    = cnn_loaders["train"],
    val_loader      = cnn_loaders["val"],
    num_epochs      = TRAIN_CFG.num_epochs,
    learning_rate   = TRAIN_CFG.learning_rate,
    weight_decay    = TRAIN_CFG.weight_decay,
    patience        = TRAIN_CFG.early_stop_patience,
    label_smoothing = TRAIN_CFG.label_smoothing,
    device          = DEVICE,
    checkpoint_path = cnn_baseline_ckpt,
    verbose         = True,
)

print(f"\nBaseline training complete.")
print(f"Best val accuracy : {baseline_history['best_val_acc']:.4f}")
print(f"Best val loss     : {baseline_history['best_val_loss']:.4f}")
print(f"Best epoch        : {baseline_history['best_epoch']}")
print(f"Training time     : {baseline_history['training_time_s']:.1f} s")


In [ ]:
def plot_training_history(history: Dict[str, List[float]], title: str = "") -> None:
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history["train_loss"]) + 1)

    # loss
    ax_loss.plot(epochs, history["train_loss"], "b-o", ms=4, label="Train loss")
    ax_loss.plot(epochs, history["val_loss"],   "r-o", ms=4, label="Val loss")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.set_title(f"Loss — {title}")
    ax_loss.legend()
    ax_loss.grid(alpha=0.2)

    # accuracy
    ax_acc.plot(epochs, history["train_acc"], "b-o", ms=4, label="Train acc")
    ax_acc.plot(epochs, history["val_acc"],   "r-o", ms=4, label="Val acc")
    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.set_title(f"Accuracy — {title}")
    ax_acc.set_ylim(0, 1)
    ax_acc.legend()
    ax_acc.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    title: str = "Confusion Matrix",
) -> None:
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046)

    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)

    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            color = "white" if cm_norm[i, j] > 0.6 else "black"
            ax.text(
                j, i,
                f"{cm[i, j]:,}\n({cm_norm[i, j]:.2f})",
                ha="center", va="center", color=color, fontsize=10
            )

    plt.tight_layout()
    plt.show()


def plot_roc_curve(
    y_true: np.ndarray,
    y_scores: np.ndarray,
    title: str = "ROC Curve",
) -> None:
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    auc = roc_auc_score(y_true, y_scores)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.4f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

print("Plot helper functions defined.")

In [ ]:
plot_training_history(baseline_history, title="Baseline FlexibleCNN")

In [ ]:
print("\nValidation set evaluation:")
criterion_eval = nn.CrossEntropyLoss()
_, val_acc, val_preds, val_labels, val_probs = evaluate_epoch(
    baseline_cnn,
    cnn_loaders["val"],
    criterion_eval,
    DEVICE,
)
val_metrics = compute_metrics(val_labels, val_preds, val_probs)
val_metrics = compute_metrics(val_labels, val_preds, val_probs)
print("\nValidation Metrics:")
pprint(val_metrics)

In [ ]:
print("\nClassification Report (Validation):")
print(classification_report(val_labels, val_preds, target_names=CLASS_NAMES))

In [ ]:
plot_confusion_matrix(val_labels, val_preds, title="Baseline CNN — Validation CM")

In [ ]:
plot_roc_curve(val_labels, val_probs, title="Baseline CNN — Validation ROC")

In [ ]:
print("\nTest set evaluation:")
_, test_acc, test_preds, test_labels, test_probs = evaluate_epoch(
    baseline_cnn,
    cnn_loaders["test"],
    criterion_eval,
    DEVICE,
)

baseline_test_metrics = compute_metrics(test_labels, test_preds, test_probs)
print("\nTest Metrics:")
pprint(baseline_test_metrics)

In [ ]:
print("\nClassification Report (Test):")
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES))

In [ ]:
plot_confusion_matrix(test_labels, test_preds, title="Baseline CNN — Test CM")

In [ ]:
plot_roc_curve(test_labels, test_probs, title="Baseline CNN — Test ROC")

## Optuna Hyperparameter Optimisation

In [ ]:
def optuna_objective(trial: optuna.Trial, device: torch.device) -> float:
    print(f"\nTrial {trial.number} started")

    n_blocks = trial.suggest_int("n_blocks", 3, 4)
    conv_configs = []
    for i in range(n_blocks):
        out_ch = trial.suggest_categorical(f"filters_{i}", [16, 32, 64, 96, 128, 256])
        k_size = trial.suggest_categorical(f"kernel_{i}", [3, 5])
        conv_configs.append((out_ch, k_size))

    dropout_rate = trial.suggest_float("dropout_rate", 0.10, 0.50)
    fc_hidden_dim = trial.suggest_categorical("fc_hidden_dim", [64, 128, 256, 512])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

    print("conv_configs   :", conv_configs)
    print("dropout_rate   :", dropout_rate)
    print("fc_hidden_dim  :", fc_hidden_dim)
    print("learning_rate  :", learning_rate)
    print("weight_decay   :", weight_decay)

    model = FlexibleCNN(
        in_channels=IMG_CFG.num_channels,
        conv_configs=conv_configs,
        dropout_rate=dropout_rate,
        fc_hidden_dim=fc_hidden_dim,
        num_classes=NUM_CLASSES,
        image_size=IMG_CFG.image_size,
    ).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=TRAIN_CFG.label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=OPTUNA_CFG.tuning_epochs,
        eta_min=1e-6,
    )

    scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

    best_val_acc = 0.0

    for epoch in range(1, OPTUNA_CFG.tuning_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=cnn_loaders["train"],
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            grad_clip=5.0,
        )

        val_loss, val_acc, _, _, _ = evaluate_epoch(
            model=model,
            loader=cnn_loaders["val"],
            criterion=criterion,
            device=device,
        )

        scheduler.step()
        best_val_acc = max(best_val_acc, val_acc)

        print(
            f"Trial {trial.number} | Epoch {epoch}/{OPTUNA_CFG.tuning_epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

        trial.report(val_acc, step=epoch)

        if trial.should_prune():
            print(f"Trial {trial.number} pruned at epoch {epoch}")
            raise optuna.TrialPruned()

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"Trial {trial.number} finished with best_val_acc={best_val_acc:.4f}")
    return best_val_acc

In [ ]:
print("Starting Optuna hyperparameter search ...")
print(f"Trials       : {OPTUNA_CFG.n_trials}")
print(f"Epochs/trial : {OPTUNA_CFG.tuning_epochs}")

sampler = optuna.samplers.TPESampler(seed=SEED)
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=2,
    n_warmup_steps=2,
    interval_steps=1,
)

study = optuna.create_study(
    direction=OPTUNA_CFG.direction,
    study_name=OPTUNA_CFG.study_name,
    sampler=sampler,
    pruner=pruner,
)

study.optimize(
    lambda trial: optuna_objective(trial, DEVICE),
    n_trials=OPTUNA_CFG.n_trials,
    show_progress_bar=True,
)

In [ ]:
best_trial = study.best_trial
print(f"\nOptuna search complete.")
print(f"Best val accuracy : {best_trial.value:.4f}")
print(f"Best trial number : {best_trial.number}")
print("\nBest hyperparameters:")
pprint(best_trial.params)

In [ ]:
# Trials dataframe
trials_df = study.trials_dataframe()
print(f"\n   Completed trials : {len(trials_df[trials_df['state']=='COMPLETE'])}")
print(f"   Pruned trials    : {len(trials_df[trials_df['state']=='PRUNED'])}")

In [ ]:
# Optuna visualisations
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
values = [t.value for t in completed]

#  Optimisation history
ax = axes[0]
if len(values) > 0:
    ax.plot(range(len(values)), values, "o-", ms=4, color="#2196F3")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Val Accuracy")
    ax.set_title("Optimisation History", fontweight="bold")

    best_idx = int(np.argmax(values))
    ax.axhline(
        values[best_idx],
        color="red",
        linestyle="--",
        alpha=0.5,
        label=f"Best: {values[best_idx]:.4f}"
    )
    ax.legend()
else:
    ax.text(0.5, 0.5, "No complete trials", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("Optimisation History", fontweight="bold")

#Parameter importance
ax2 = axes[1]
try:
    importance = optuna.importance.get_param_importances(study)

    keys = list(importance.keys())[:8]
    vals = [importance[k] for k in keys]

    keys = keys[::-1]
    vals = vals[::-1]

    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(keys)))
    ax2.barh(keys, vals, color=colors)
    ax2.set_title("Parameter Importance", fontweight="bold")
    ax2.set_xlabel("Importance score")
except Exception:
    ax2.text(
        0.5, 0.5,
        "Importance not available\n(need >=2 complete trials)",
        ha="center", va="center",
        transform=ax2.transAxes
    )
    ax2.set_title("Parameter Importance", fontweight="bold")

#Trial value distribution
ax3 = axes[2]
if len(values) > 0:
    ax3.hist(values, bins=min(15, len(values)), color="#4CAF50", edgecolor="white")
    ax3.set_xlabel("Val Accuracy")
    ax3.set_ylabel("Count")
    ax3.set_title("Trial Value Distribution", fontweight="bold")
else:
    ax3.text(0.5, 0.5, "No values to plot", ha="center", va="center", transform=ax3.transAxes)
    ax3.set_title("Trial Value Distribution", fontweight="bold")

plt.suptitle("Optuna Study Results", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
#  best CNN from Optuna params
best_trial = study.best_trial
best_params = best_trial.params

best_n_blocks = best_params["n_blocks"]
best_conv_cfg = [
    (best_params[f"filters_{i}"], best_params[f"kernel_{i}"])
    for i in range(best_n_blocks)]

In [ ]:
best_cnn = FlexibleCNN(
    in_channels   = IMG_CFG.num_channels,
    conv_configs  = best_conv_cfg,
    dropout_rate  = best_params["dropout_rate"],
    fc_hidden_dim = best_params["fc_hidden_dim"],
    num_classes   = NUM_CLASSES,
    image_size    = IMG_CFG.image_size,
)

In [ ]:
best_cnn.describe()

In [ ]:
print("\n" + "=" * 65)
print("  Training Optimised FlexibleCNN (Full Run)")
print("=" * 65)

optimised_cnn_ckpt = PATH_CFG.checkpoint_dir / "cnn_optimised.pt"

best_cnn, optimised_history = train_model(
    model           = best_cnn,
    train_loader    = cnn_loaders["train"],
    val_loader      = cnn_loaders["val"],
    num_epochs      = TRAIN_CFG.num_epochs,
    learning_rate   = best_params["learning_rate"],
    weight_decay    = best_params["weight_decay"],
    patience        = TRAIN_CFG.early_stop_patience,
    label_smoothing = TRAIN_CFG.label_smoothing,
    device          = DEVICE,
    checkpoint_path = optimised_cnn_ckpt,
    verbose         = True,
)

print(f"\nOptimised CNN training complete.")
print(f"Best val accuracy : {optimised_history['best_val_acc']:.4f}")
print(f"Best val loss     : {optimised_history['best_val_loss']:.4f}")
print(f"Best epoch        : {optimised_history['best_epoch']}")
print(f"Training time     : {optimised_history['training_time_s']:.1f} s")

plot_training_history(optimised_history, title="Optimised FlexibleCNN")

In [ ]:
criterion_eval = nn.CrossEntropyLoss()

_, _, opt_preds, opt_labels, opt_probs = evaluate_epoch(
    best_cnn,
    cnn_loaders["test"],
    criterion_eval,
    DEVICE,
)

optimised_test_metrics = compute_metrics(opt_labels, opt_preds, opt_probs)

In [ ]:
print("\nOptimised CNN — Test Metrics:")
pprint(optimised_test_metrics)

In [ ]:
print("\nClassification Report (Test):")
print(classification_report(opt_labels, opt_preds, target_names=CLASS_NAMES))

In [ ]:
plot_confusion_matrix(opt_labels, opt_preds, title="Optimised CNN — Test CM")

In [ ]:
plot_roc_curve(opt_labels, opt_probs, title="Optimised CNN — Test ROC")

## PreTrained ResNet Models

In [ ]:
from tqdm.auto import tqdm

In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = TRAIN_CFG.num_epochs,
    learning_rate: float = TRAIN_CFG.learning_rate,
    weight_decay: float = TRAIN_CFG.weight_decay,
    patience: int = TRAIN_CFG.early_stop_patience,
    label_smoothing: float = TRAIN_CFG.label_smoothing,
    device: torch.device = DEVICE,
    checkpoint_path: Optional[Path] = None,
    verbose: bool = True,
    grad_clip: Optional[float] = 5.0,
) -> Tuple[nn.Module, Dict[str, Any]]:

    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=1e-6,
    )

    scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = 1
    patience_ctr = 0
    t_start = time.time()

    epoch_bar = tqdm(range(1, num_epochs + 1), desc="Training", leave=True)

    for epoch in epoch_bar:
        tr_loss, tr_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            grad_clip=grad_clip,
        )

        vl_loss, vl_acc, _, _, _ = evaluate_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        lr_now = scheduler.get_last_lr()[0]

        epoch_bar.set_postfix({
            "train_loss": f"{tr_loss:.4f}",
            "train_acc": f"{tr_acc:.4f}",
            "val_loss": f"{vl_loss:.4f}",
            "val_acc": f"{vl_acc:.4f}",
            "lr": f"{lr_now:.2e}",
        })

        if verbose:
            print(
                f"[Epoch {epoch:>2}/{num_epochs}] "
                f"train_loss={tr_loss:.4f} | train_acc={tr_acc:.4f} | "
                f"val_loss={vl_loss:.4f} | val_acc={vl_acc:.4f} | "
                f"lr={lr_now:.2e}"
            )

        if vl_loss < best_val_loss - 1e-4:
            best_val_loss = vl_loss
            best_val_acc = vl_acc
            best_epoch = epoch
            patience_ctr = 0

            if checkpoint_path is not None:
                torch.save(model.state_dict(), checkpoint_path)
        else:
            patience_ctr += 1

            if patience_ctr >= patience:
                if verbose:
                    print(
                        f"Early stop at epoch {epoch} "
                        f"(best val loss={best_val_loss:.4f}, "
                        f"best val acc={best_val_acc:.4f}, "
                        f"best epoch={best_epoch})"
                    )
                break

    if checkpoint_path is not None and checkpoint_path.exists():
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    history["best_val_loss"] = best_val_loss
    history["best_val_acc"] = best_val_acc
    history["best_epoch"] = best_epoch
    history["training_time_s"] = time.time() - t_start

    return model, history

In [ ]:
def build_resnet(
    architecture: str,
    num_classes: int,
    pretrained: bool = True,
) -> nn.Module:
    weight_map = {
        "resnet18": models.ResNet18_Weights.DEFAULT,
        "resnet34": models.ResNet34_Weights.DEFAULT,
        "resnet50": models.ResNet50_Weights.DEFAULT,
    }

    builder_map = {
        "resnet18": models.resnet18,
        "resnet34": models.resnet34,
        "resnet50": models.resnet50,
    }

    assert architecture in builder_map, f"Unknown architecture: {architecture}"

    weights = weight_map[architecture] if pretrained else None
    model = builder_map[architecture](weights=weights)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    return model


def train_and_evaluate_resnet(
    architecture: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    num_classes: int = NUM_CLASSES,
    num_epochs: int = TRAIN_CFG.num_epochs,
    learning_rate: float = 1e-4,
    weight_decay: float = TRAIN_CFG.weight_decay,
    patience: int = TRAIN_CFG.early_stop_patience,
    device: torch.device = DEVICE,
    checkpoint_dir: Path = PATH_CFG.checkpoint_dir,
) -> Tuple[nn.Module, Dict, Dict, Dict]:

    print("\n" + "=" * 65)
    print(f"  Fine-tuning {architecture.upper()}")
    print("=" * 65)

    model = build_resnet(architecture, num_classes)
    ckpt = checkpoint_dir / f"{architecture}.pt"

    model, history = train_model(
        model           = model,
        train_loader    = train_loader,
        val_loader      = val_loader,
        num_epochs      = num_epochs,
        learning_rate   = learning_rate,
        weight_decay    = weight_decay,
        patience        = patience,
        label_smoothing = TRAIN_CFG.label_smoothing,
        device          = device,
        checkpoint_path = ckpt,
        verbose         = True,
    )

    criterion = nn.CrossEntropyLoss()

    _, _, v_preds, v_labels, v_probs = evaluate_epoch(
        model,
        val_loader,
        criterion,
        device,
    )
    val_metrics = compute_metrics(v_labels, v_preds, v_probs)

    _, _, t_preds, t_labels, t_probs = evaluate_epoch(
        model,
        test_loader,
        criterion,
        device,
    )
    test_metrics = compute_metrics(t_labels, t_preds, t_probs)

    print(f"\n{architecture.upper()} — Validation Metrics:")
    pprint(val_metrics)

    print(f"\n{architecture.upper()} — Test Metrics:")
    pprint(test_metrics)

    print("\nClassification Report (Test):")
    print(classification_report(t_labels, t_preds, target_names=CLASS_NAMES))

    plot_training_history(history, title=architecture.upper())
    plot_confusion_matrix(t_labels, t_preds, title=f"{architecture.upper()} — Test CM")
    plot_roc_curve(t_labels, t_probs, title=f"{architecture.upper()} — Test ROC")

    return model, history, val_metrics, test_metrics

In [ ]:
resnet18_model, resnet18_history, resnet18_val, resnet18_test = train_and_evaluate_resnet(
    architecture="resnet18",
    train_loader=resnet_loaders["train"],
    val_loader=resnet_loaders["val"],
    test_loader=resnet_loaders["test"],
    num_epochs=TRAIN_CFG.num_epochs,
    learning_rate=1e-4,
)

In [ ]:
print(f"\nResnet training complete.")
print(f"Best val accuracy : {resnet18_history['best_val_acc']:.4f}")
print(f"Best val loss     : {resnet18_history['best_val_loss']:.4f}")
print(f"Best epoch        : {resnet18_history['best_epoch']}")
print(f"Training time     : {resnet18_history['training_time_s']:.1f} s")

In [ ]:
resnet34_model, resnet34_history, resnet34_val, resnet34_test = train_and_evaluate_resnet(
    architecture="resnet34",
    train_loader=resnet_loaders["train"],
    val_loader=resnet_loaders["val"],
    test_loader=resnet_loaders["test"],
    num_epochs=TRAIN_CFG.num_epochs,
    learning_rate=1e-4,
)

In [ ]:
def get_model_size_mb(model: nn.Module) -> float:
    total_bytes = sum(
        p.numel() * p.element_size()
        for p in list(model.parameters()) + list(model.buffers())
    )
    return total_bytes / (1024 ** 2)

In [ ]:
@torch.no_grad()
def measure_inference_time_ms(
    model          : nn.Module,
    input_tensor   : torch.Tensor,
    num_iterations : int = EFFICIENCY_CFG.benchmark_iterations,
    warmup         : int = EFFICIENCY_CFG.warmup_iterations,
    device         : torch.device = DEVICE, ) -> float:

    model.eval()
    model.to(device)
    input_tensor = input_tensor.to(device)

    # Warm-up
    for _ in range(warmup):
        _ = model(input_tensor)

    if device.type == "cuda":
        torch.cuda.synchronize()

    # timed benchmark
    t_start = time.perf_counter()

    for _ in range(num_iterations):
        _ = model(input_tensor)

    if device.type == "cuda":
        torch.cuda.synchronize()

    t_end = time.perf_counter()

    avg_ms = (t_end - t_start) * 1000 / num_iterations
    return avg_ms


In [ ]:
def build_efficiency_table(
    models_dict        : Dict[str, nn.Module],
    test_metrics_dict  : Dict[str, Dict],
    cnn_input_size     : int = IMG_CFG.image_size,
    resnet_input_size  : int = RESNET_IMG_SIZE, ) -> pd.DataFrame:
    rows = []

    for model_name, model in models_dict.items():
        try:
            size_mb = get_model_size_mb(model)

            is_resnet = "resnet" in model_name.lower()
            img_size  = resnet_input_size if is_resnet else cnn_input_size

            dummy_img = torch.zeros(1, 3, img_size, img_size)
            inf_ms    = measure_inference_time_ms(model, dummy_img)

            metrics = test_metrics_dict.get(model_name, {})

            row = {
                "model_name"    : model_name,
                "accuracy"      : metrics.get("accuracy",  float("nan")),
                "precision"     : metrics.get("precision", float("nan")),
                "recall"        : metrics.get("recall",    float("nan")),
                "f1"            : metrics.get("f1",        float("nan")),
                "roc_auc"       : metrics.get("roc_auc",   float("nan")),
                "model_size_mb" : round(size_mb, 3),
                "inference_ms"  : round(inf_ms, 4),
            }

            rows.append(row)

            print(
                f"{model_name:30s} | "
                f"acc={row['accuracy']:.4f} | "
                f"size={size_mb:.2f} MB | "
                f"lat={inf_ms:.3f} ms"
            )

        except Exception as e:
            print(f"Error processing {model_name}: {e}")

    return pd.DataFrame(rows)

In [ ]:
all_models = {
    "Custom_CNN_Baseline"  : baseline_cnn,
    "Custom_CNN_Optimised" : best_cnn,
    "ResNet18"             : resnet18_model,
    "ResNet34"             : resnet34_model,
}

In [ ]:
all_test_metrics = {
    "Custom_CNN_Baseline"  : baseline_test_metrics,
    "Custom_CNN_Optimised" : optimised_test_metrics,
    "ResNet18"             : resnet18_test,
    "ResNet34"             : resnet34_test,
}
print("\nComputing efficiency metrics for all models …\n")
efficiency_df = build_efficiency_table(all_models, all_test_metrics)

In [ ]:
print("\n📊 Full Comparison Table:")

display_cols = [
    "model_name",
    "accuracy",
    "f1",
    "roc_auc",
    "model_size_mb",
    "inference_ms"
]

print(efficiency_df[display_cols].to_string(index=False))

In [ ]:
def plot_efficiency_comparison(df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))

    # Normalize values for plotting on the same scale if needed, or use different axes
    # For simplicity, we'll plot them as is and focus on relative comparisons.

    # Create a scatter plot comparing accuracy vs. inference time, with model size as marker size
    scatter = ax.scatter(
        x=df["inference_ms"],
        y=df["accuracy"],
        s=df["model_size_mb"] * 50, # Scale marker size for visibility
        alpha=0.7,
        c=df["f1"], # Color by F1-score
        cmap="viridis",
        edgecolors="w",
        linewidths=0.5
    )

    # Annotate points with model names
    for i, row in df.iterrows():
        ax.text(row["inference_ms"], row["accuracy"], row["model_name"], fontsize=9, ha='right')

    ax.set_xlabel("Inference Time (ms)")
    ax.set_ylabel("Accuracy")
    ax.set_title("Model Efficiency: Accuracy vs. Inference Time (Size coded by marker size)")
    ax.grid(True, linestyle='--', alpha=0.6)

    # Add a colorbar for F1-score
    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label("F1-Score")

    plt.tight_layout()
    plt.show()

plot_efficiency_comparison(efficiency_df)

In [ ]:
def select_best_model_weighted(
    df      : pd.DataFrame,
    weights : Optional[Dict[str, float]] = None, ) -> Tuple[str, pd.DataFrame]:

    if weights is None:
        weights = {
            "accuracy"      : 0.40,
            "f1"            : 0.25,
            "roc_auc"       : 0.15,
            "model_size_mb" : 0.10,
            "inference_ms"  : 0.10,
        }

    scored = df.copy()

    for metric, w in weights.items():
        if metric not in scored.columns:
            raise ValueError(f"Missing required metric column: {metric}")

        col = scored[metric]
        col_min, col_max = col.min(), col.max()

        if col_max - col_min < 1e-9:
            # normalise to 0.5
            normalised = pd.Series(0.5, index=col.index)
        else:
            normalised = (col - col_min) / (col_max - col_min)

        # higher score = better
        if metric in ("model_size_mb", "inference_ms"):
            normalised = 1.0 - normalised

        scored[f"score_{metric}"] = normalised * w

    score_cols = [c for c in scored.columns if c.startswith("score_")]
    scored["weighted_score"] = scored[score_cols].sum(axis=1)

    best_row  = scored.loc[scored["weighted_score"].idxmax()]
    best_name = best_row["model_name"]

    return best_name, scored.sort_values("weighted_score", ascending=False)

In [ ]:
def select_best_model_constrained(
    df               : pd.DataFrame,
    max_size_mb      : float,
    max_inference_ms : float, ) -> Tuple[Optional[str], pd.DataFrame]:

    viable = df[
        (df["model_size_mb"] <= max_size_mb) &
        (df["inference_ms"] <= max_inference_ms)
    ].copy()

    if viable.empty:
        print(
            f"No model satisfies constraints "
            f"(size ≤ {max_size_mb} MB, latency ≤ {max_inference_ms} ms)."
        )
        return None, viable

    best_row  = viable.loc[viable["accuracy"].idxmax()]
    best_name = best_row["model_name"]

    return best_name, viable.sort_values("accuracy", ascending=False)

In [ ]:
print("\nWeighted Score Selection")
print("-" * 50)

weighted_winner, scored_df = select_best_model_weighted(efficiency_df)

print(
    scored_df[
        [
            "model_name",
            "accuracy",
            "f1",
            "roc_auc",
            "model_size_mb",
            "inference_ms",
            "weighted_score",
        ]
    ].to_string(index=False)
)

print(f"\nWeighted winner : {weighted_winner}")

In [ ]:
print("\nConstraint-Based Selection")
print("-" * 50)

MAX_SIZE_MB = 50.0
MAX_INFERENCE_MS = 20.0

constrained_winner, viable_df = select_best_model_constrained(
    efficiency_df,
    MAX_SIZE_MB,
    MAX_INFERENCE_MS,
)

if not viable_df.empty:
    print(
        viable_df[
            ["model_name", "accuracy", "model_size_mb", "inference_ms"]
        ].to_string(index=False)
    )

print(f"\nConstrained winner : {constrained_winner}")

In [ ]:
print("=" * 70)
print("FINAL MODEL ANALYSIS REPORT")
print("=" * 70)

best_acc_row = efficiency_df.loc[efficiency_df["accuracy"].idxmax()]
pure_perf_winner = best_acc_row["model_name"]

print(f"\n1.Best by pure accuracy     : {pure_perf_winner}")
print(f"Accuracy   : {best_acc_row['accuracy']:.4f}")
print(f"F1         : {best_acc_row['f1']:.4f}")
print(f"ROC-AUC    : {best_acc_row['roc_auc']:.4f}")

print(f"\n2. Best by weighted criteria : {weighted_winner}")
row = scored_df[scored_df["model_name"] == weighted_winner].iloc[0]
print(f"Weighted score : {row['weighted_score']:.4f}")
print(f"Accuracy       : {row['accuracy']:.4f}")
print(f"Model size     : {row['model_size_mb']:.2f} MB")
print(f"Inference      : {row['inference_ms']:.3f} ms")

print(f"\n3. Best under deployment constraints")
print(f"(≤{MAX_SIZE_MB} MB, ≤{MAX_INFERENCE_MS} ms)")
print(f"Winner : {constrained_winner if constrained_winner is not None else 'No valid model'}")

In [ ]:
#Full comparison table
print("\n📊 Full Model Comparison:")
print(
    efficiency_df[
        [
            "model_name",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "model_size_mb",
            "inference_ms",
        ]
    ].to_string(index=False)
)

In [ ]:
FINAL_WINNER_NAME = weighted_winner
final_model       = all_models[FINAL_WINNER_NAME]
final_metrics     = all_test_metrics[FINAL_WINNER_NAME]

print(f"Saving final model: {FINAL_WINNER_NAME}")

In [ ]:
PATH_CFG.checkpoint_dir.mkdir(parents=True, exist_ok=True)
PATH_CFG.results_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
#checkpoint dict
checkpoint = {
    "model_name"   : FINAL_WINNER_NAME,
    "state_dict"   : final_model.state_dict(),
    "class_to_idx" : CLASS_TO_IDX,
    "idx_to_class" : IDX_TO_CLASS,
    "test_metrics" : final_metrics,
    "image_size"   : (
        RESNET_IMG_SIZE
        if "resnet" in FINAL_WINNER_NAME.lower()
        else IMG_CFG.image_size
    ),
    "normalisation": {
        "mean": (
            IMG_CFG.imagenet_mean
            if "resnet" in FINAL_WINNER_NAME.lower()
            else IMG_CFG.cifake_mean
        ),
        "std": (
            IMG_CFG.imagenet_std
            if "resnet" in FINAL_WINNER_NAME.lower()
            else IMG_CFG.cifake_std
        ),
    },
}

In [ ]:
# Include architecture config for custom CNN
if hasattr(final_model, "config"):
    checkpoint["model_config"] = final_model.config

# Include Optuna best params if relevant
if "Optimised" in FINAL_WINNER_NAME:
    checkpoint["best_hyperparams"] = best_params

In [ ]:
final_ckpt_path = PATH_CFG.checkpoint_dir / "best_model.pt"
torch.save(checkpoint, final_ckpt_path)
print(f"Model checkpoint saved → {final_ckpt_path}")

In [ ]:
csv_path = PATH_CFG.results_dir / "model_comparison.csv"
efficiency_df.to_csv(csv_path, index=False)
print(f"Results CSV saved → {csv_path}")

In [ ]:
# For verifying the checkpoint can be re-loaded
loaded_ckpt = torch.load(final_ckpt_path, map_location="cpu", weights_only=False)

print(f"\nCheckpoint verified:")
print(f"keys      : {list(loaded_ckpt.keys())}")
print(f"Class map : {loaded_ckpt['class_to_idx']}")

if "test_metrics" in loaded_ckpt and "accuracy" in loaded_ckpt["test_metrics"]:
    print(f"   Test acc  : {loaded_ckpt['test_metrics']['accuracy']:.4f}")

#Inference Helper

In [ ]:
def predict_image(
    image_input     : Any,
    checkpoint_path : Path = Path("checkpoints/best_model.pt"),
    device          : torch.device = DEVICE, ) -> Dict[str, Any]:

    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model_name = ckpt.get("model_name", "")

    if "resnet" in model_name.lower():
        architecture = "resnet18" if "18" in model_name else "resnet34"
        model = build_resnet(architecture, num_classes=2, pretrained=False)
    else:
        cfg = ckpt["model_config"]
        model = FlexibleCNN(**{k: v for k, v in cfg.items() if k != "flat_dim"})

    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    model.to(device)

    img_size = ckpt["image_size"]
    mean     = ckpt["normalisation"]["mean"]
    std      = ckpt["normalisation"]["std"]

    resize_size = tuple(img_size) if isinstance(img_size, (tuple, list)) else (img_size, img_size)

    infer_tf = transforms.Compose([
        transforms.Resize(resize_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    if isinstance(image_input, (str, Path)):
        img = Image.open(image_input).convert("RGB")
    else:
        img = image_input.convert("RGB")

    tensor = infer_tf(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]

    idx_to_class = ckpt.get("idx_to_class", {0: "REAL", 1: "FAKE"})
    idx_to_class = {int(k): v for k, v in idx_to_class.items()}

    pred_idx   = probs.argmax().item()
    pred_label = idx_to_class[pred_idx]
    confidence = probs[pred_idx].item()

    return {
        "label"      : pred_label,
        "label_idx"  : pred_idx,
        "confidence" : confidence,
        "probabilities": {
            idx_to_class[i]: probs[i].item()
            for i in range(len(probs))
        },
    }

In [ ]:
# let's run inference on a sample from the test set
sample_path, sample_label = test_samples[42]

result = predict_image(
    image_input     = sample_path,
    checkpoint_path = Path("checkpoints/best_model.pt"),
    device          = DEVICE,
)

print("\nExample Inference")
print(f"Image path   : {sample_path.name}")
print(f"True label   : {IDX_TO_CLASS[sample_label]}")
print(f"Predicted    : {result['label']}  (confidence: {result['confidence']:.4f})")
print(f"Probabilities: {result['probabilities']}")
print(f"Correct      : {'V' if result['label_idx'] == sample_label else 'X'}")

In [ ]:
# Show the image
fig, ax = plt.subplots(figsize=(3, 3))
img = Image.open(sample_path).convert("RGB")
ax.imshow(img)
ax.set_title(
    f"True: {IDX_TO_CLASS[sample_label]} | Pred: {result['label']}\n"
    f"Conf: {result['confidence']:.3f}",
    fontsize=10,
)
ax.axis("off")
plt.tight_layout()
plt.show()

# Error Analysis

In [ ]:
def collect_errors(
    model            : nn.Module,
    test_loader      : DataLoader,
    test_samples_list: List[Tuple[Path, int]],
    device           : torch.device,
    max_errors       : int = 10,
) -> Dict[str, List]:

    model.eval()
    model.to(device)

    errors = {
        "false_positive": [],  # REAL -> FAKE
        "false_negative": [],  # FAKE -> REAL
    }

    sample_idx = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            probs  = torch.softmax(logits, dim=1)
            preds  = logits.argmax(dim=1)

            for i in range(len(labels)):
                true_lbl = labels[i].item()
                pred_lbl = preds[i].item()
                conf     = probs[i, pred_lbl].item()

                if true_lbl != pred_lbl:
                    global_idx = sample_idx + i

                    if global_idx < len(test_samples_list):
                        path = test_samples_list[global_idx][0]

                        category = (
                            "false_positive" if true_lbl == 0 else "false_negative"
                        )

                        if len(errors[category]) < max_errors:
                            errors[category].append(
                                (path, true_lbl, pred_lbl, conf)
                            )

            sample_idx += len(labels)

            if all(len(v) >= max_errors for v in errors.values()):
                break

    return errors


In [ ]:
def show_error_grid(errors: Dict[str, List], n_show: int = 5) -> None:
    fig, axes = plt.subplots(2, n_show, figsize=(2.5 * n_show, 5))

    if n_show == 1:
        axes = np.array(axes).reshape(2, 1)

    titles = {
        "false_positive": "False Positive (Real → Fake)",
        "false_negative": "False Negative (Fake → Real)",
    }

    categories = ["false_positive", "false_negative"]

    for row_idx, category in enumerate(categories):
        samples_list = errors.get(category, [])

        for col_idx in range(n_show):
            ax = axes[row_idx][col_idx]

            if col_idx < len(samples_list):
                path, true_lbl, pred_lbl, conf = samples_list[col_idx]

                try:
                    img = Image.open(path).convert("RGB")
                    ax.imshow(img)
                    ax.set_title(
                        f"T:{IDX_TO_CLASS[true_lbl]}\n"
                        f"P:{IDX_TO_CLASS[pred_lbl]}\n"
                        f"{conf:.2f}",
                        fontsize=7,
                        color="red",
                    )
                except Exception:
                    ax.text(
                        0.5, 0.5, "N/A",
                        ha="center", va="center",
                        transform=ax.transAxes,
                    )

            ax.axis("off")

        axes[row_idx][0].set_ylabel(
            titles[category],
            fontsize=9,
            rotation=0,
            labelpad=80,
            va="center",
        )

    plt.suptitle("Misclassified Samples (Best Model)", fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
# run error analysis on the best model
# will use whichever loader matches the winning model
is_resnet = "resnet" in FINAL_WINNER_NAME.lower()
err_loader = resnet_loaders["test"] if is_resnet else cnn_loaders["test"]

errors = collect_errors(
    final_model,
    err_loader,
    test_samples,
    DEVICE,
    max_errors=5,
)

total_errors = sum(len(v) for v in errors.values())

print(f"\nError Analysis — {FINAL_WINNER_NAME}")
print(f"False positives (Real→Fake) : {len(errors['false_positive'])}")
print(f"False negatives (Fake→Real) : {len(errors['false_negative'])}")
print(f"Total shown                 : {total_errors}")

In [ ]:
show_error_grid(errors, n_show=5)